In [1]:
# =============================================================
# REGRESSION ASSIGNMENT - Insurance Charges Prediction
# =============================================================

# PROBLEM STATEMENT:
# A client wants to predict insurance charges for individuals based on personal attributes such as age, BMI, number of children, sex and 
# smoking status. We need to build the best possible regression model that can predict these charges accurately.

# APPROACH:
# We will try 5 algorithms:

#   1. Multiple Linear Regression (with all features)
#   2. Support Vector Regression (SVR)
#   3. Decision Tree Regressor
#   4. Random Forest Regressor
# Each model will be hyperparameter tuned. The best model across all algorithms will be saved and used for final prediction.

print("Assignment: Insurance Charges Prediction")
print("Algorithms: Multiple linear regression ,SVR, Decision Tree, Random Forest")

Assignment: Insurance Charges Prediction
Algorithms: Multiple linear regression ,SVR, Decision Tree, Random Forest


In [2]:
# Step 1: Import all required libraries
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pickle

In [3]:
# Step 2: Load the dataset
# Columns: age, sex, bmi, children, smoker, charges (target)
dataset = pd.read_csv("insurance_pre.csv")

In [4]:
# Step 3: View the dataset
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [5]:
# Step 4: Basic information about the dataset
# Input features: age, sex, bmi, children, smoker
# Target variable (what we predict): charges
print("Shape:", dataset.shape)
print("\nData Types:")
print(dataset.dtypes)
print("\nBasic Statistics:")
dataset.describe()

Shape: (1338, 6)

Data Types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
charges     float64
dtype: object

Basic Statistics:


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [6]:
# Step 5: Check for missing values
dataset.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
charges     0
dtype: int64

In [7]:
# Step 6: Pre-processing - Convert categorical (text) columns to numbers
# 'sex'    has 2 values: male / female  -> we encode this as sex_male (1=male, 0=female)
# 'smoker' has 2 values: yes / no       -> we encode this as smoker_yes (1=yes, 0=no)
# We use One-Hot Encoding with drop_first=True to avoid redundancy
# dtype=int ensures we get 0 and 1 instead of True/False
dataset = pd.get_dummies(dataset, drop_first=True, dtype=int)
print("Columns after encoding:", dataset.columns.tolist())

Columns after encoding: ['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes']


In [8]:
# Step 7: View dataset after encoding
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [9]:
# Step 8: Separate Input (X) features and Output (y) target
# Input features (5 columns): age, bmi, children, sex_male, smoker_yes
# Target (what we predict): charges
X = dataset.drop("charges", axis=1)
y = dataset["charges"]
print("Input features:", X.columns.tolist())
print("Target: charges")

Input features: ['age', 'bmi', 'children', 'sex_male', 'smoker_yes']
Target: charges


In [10]:
# Step 9: Split data into Train (70%) and Test (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 936
Testing samples: 402


In [11]:
# Step 10: Feature Scaling for SVR
# SVR is sensitive to the scale of data so we standardize the features
# We fit the scaler only on training data and transform both train and test
# This scaler is also saved so we can use it during deployment
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

In [12]:
# =============================================================
# MODEL 1: LINEAR REGRESSION (Simple & Multiple)
# =============================================================
# Linear Regression finds the best straight line relationship
# between the input features and the target (charges) When using all 5 features it is called Multiple Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
score_lr = round(r2_score(y_test, y_pred_lr), 4)
print("Linear Regression R2 Score:", score_lr)
print("Coefficients:", lr.coef_)
print("Intercept:", lr.intercept_)

Linear Regression R2 Score: 0.7895
Coefficients: [  257.8006705    321.06004271   469.58113407   -41.74825718
 23418.6671912 ]
Intercept: -12057.24484599853


In [13]:
# Save the best Linear Regression model
pickle.dump(lr, open("best_linear_regression.sav", "wb"))
print("Linear Regression model saved.")

Linear Regression model saved.


In [14]:
# =============================================================
# MODEL 2: SUPPORT VECTOR REGRESSION (SVR) - Hyperparameter Tuning
# =============================================================

# --- Experiment 1: kernel tuning ---
# kernel defines the type of decision boundary SVR learns
print("=== SVR Experiment 1: Kernel Tuning (C=1000) ===")
for k in ['linear', 'rbf', 'poly', 'sigmoid']:
    m = SVR(kernel=k, C=1000)
    m.fit(X_train_sc, y_train)
    s = round(r2_score(y_test, m.predict(X_test_sc)), 4)
    print(f"  kernel={k} -> R2 = {s}")

=== SVR Experiment 1: Kernel Tuning (C=1000) ===
  kernel=linear -> R2 = 0.7649
  kernel=rbf -> R2 = 0.8102
  kernel=poly -> R2 = 0.8566
  kernel=sigmoid -> R2 = 0.2875


In [15]:
# --- Experiment 2: C tuning for best kernel (poly) ---
# C controls how tightly the model fits the training data
# Higher C = tighter fit but risks overfitting
print("=== SVR Experiment 2: C Tuning (kernel=poly) ===")
for c in [100, 1000, 5000, 10000, 50000]:
    m = SVR(kernel='poly', C=c)
    m.fit(X_train_sc, y_train)
    s = round(r2_score(y_test, m.predict(X_test_sc)), 4)
    print(f"  C={c} -> R2 = {s}")

=== SVR Experiment 2: C Tuning (kernel=poly) ===
  C=100 -> R2 = 0.618
  C=1000 -> R2 = 0.8566
  C=5000 -> R2 = 0.8596
  C=10000 -> R2 = 0.8592
  C=50000 -> R2 = 0.8576


In [16]:
# --- Experiment 3: epsilon tuning ---
# epsilon is the tolerance tube - errors inside this range are ignored
print("=== SVR Experiment 3: Epsilon Tuning (kernel=rbf, C=10000) ===")
for e in [0.01, 0.1, 0.5, 1.0, 5.0]:
    m = SVR(kernel='poly', C=10000, epsilon=e)
    m.fit(X_train_sc, y_train)
    s = round(r2_score(y_test, m.predict(X_test_sc)), 4)
    print(f"  epsilon={e} -> R2 = {s}")

=== SVR Experiment 3: Epsilon Tuning (kernel=rbf, C=10000) ===
  epsilon=0.01 -> R2 = 0.8592
  epsilon=0.1 -> R2 = 0.8592
  epsilon=0.5 -> R2 = 0.8592
  epsilon=1.0 -> R2 = 0.8591
  epsilon=5.0 -> R2 = 0.8591


In [17]:
# Best SVR Model: kernel=rbf, C=10000 , epsilon=0.1
best_svr = SVR(kernel='rbf', C=10000, epsilon=0.1)
best_svr.fit(X_train_sc, y_train)
score_svr = round(r2_score(y_test, best_svr.predict(X_test_sc)), 4)
print("Best SVR R2 Score:", score_svr)

# Save best SVR model and the scaler
pickle.dump(best_svr, open("best_svr.sav", "wb"))
pickle.dump(sc, open("scaler.sav", "wb"))
print("SVR model and scaler saved.")

Best SVR R2 Score: 0.878
SVR model and scaler saved.


In [18]:
# =============================================================
# MODEL 3: DECISION TREE REGRESSOR - Hyperparameter Tuning
# =============================================================

# --- Experiment 1: criterion ---
print("=== DT Experiment 1: Criterion Tuning ===")
for crit in ['squared_error', 'friedman_mse', 'absolute_error', 'poisson']:
    m = DecisionTreeRegressor(criterion=crit, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  criterion={crit} -> R2 = {s}")

=== DT Experiment 1: Criterion Tuning ===
  criterion=squared_error -> R2 = 0.6957
  criterion=friedman_mse -> R2 = 0.7115
  criterion=absolute_error -> R2 = 0.6681
  criterion=poisson -> R2 = 0.7288


In [19]:
# --- Experiment 2: min_samples_split ---
# Controls how many samples are needed before a node can split further
print("=== DT Experiment 2: min_samples_split (criterion=poisson) ===")
for mss in [2, 3, 5, 7, 10, 15, 20]:
    m = DecisionTreeRegressor(criterion='poisson', min_samples_split=mss, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  min_samples_split={mss} -> R2 = {s}")

=== DT Experiment 2: min_samples_split (criterion=poisson) ===
  min_samples_split=2 -> R2 = 0.7288
  min_samples_split=3 -> R2 = 0.7447
  min_samples_split=5 -> R2 = 0.7937
  min_samples_split=7 -> R2 = 0.7962
  min_samples_split=10 -> R2 = 0.8036
  min_samples_split=15 -> R2 = 0.8083
  min_samples_split=20 -> R2 = 0.83


In [20]:
# --- Experiment 3: max_features ---
# How many features the tree considers at each split
print("=== DT Experiment 3: max_features (criterion=poisson, mss=20) ===")
for mf in [None, 'sqrt', 'log2', 3, 4, 5]:
    m = DecisionTreeRegressor(criterion='poisson', min_samples_split=20, max_features=mf, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  max_features={mf} -> R2 = {s}")

=== DT Experiment 3: max_features (criterion=poisson, mss=20) ===
  max_features=None -> R2 = 0.83
  max_features=sqrt -> R2 = 0.8299
  max_features=log2 -> R2 = 0.8299
  max_features=3 -> R2 = 0.8296
  max_features=4 -> R2 = 0.8095
  max_features=5 -> R2 = 0.83


In [21]:
# --- Experiment 4: max_depth ---
print("=== DT Experiment 4: max_depth ===")
for d in [3, 5, 7, 10, None]:
    m = DecisionTreeRegressor(criterion='poisson', min_samples_split=20, max_depth=d, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  max_depth={d} -> R2 = {s}")

=== DT Experiment 4: max_depth ===
  max_depth=3 -> R2 = 0.8778
  max_depth=5 -> R2 = 0.8397
  max_depth=7 -> R2 = 0.8409
  max_depth=10 -> R2 = 0.8334
  max_depth=None -> R2 = 0.83


In [22]:
# Best Decision Tree: criterion=absolute_error, min_samples_split=10
best_dt = DecisionTreeRegressor(criterion='poisson', min_samples_split=20, max_depth=3, random_state=0)
best_dt.fit(X_train, y_train)
score_dt = round(r2_score(y_test, best_dt.predict(X_test)), 4)
print("Best Decision Tree R2 Score:", score_dt)

pickle.dump(best_dt, open("best_decision_tree.sav", "wb"))
print("Decision Tree model saved.")

Best Decision Tree R2 Score: 0.8778
Decision Tree model saved.


In [23]:
# =============================================================
# MODEL 4: RANDOM FOREST REGRESSOR - Hyperparameter Tuning
# =============================================================
# Random Forest builds many decision trees and averages their predictions this reduces overfitting compared to a single decision tree

# --- Experiment 1: n_estimators ---
# n_estimators = number of trees in the forest
# More trees = more stable predictions but slower to train
print("=== RF Experiment 1: n_estimators Tuning ===")
for n in [10, 50, 100, 200, 300]:
    m = RandomForestRegressor(n_estimators=n, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  n_estimators={n} -> R2 = {s}")

=== RF Experiment 1: n_estimators Tuning ===
  n_estimators=10 -> R2 = 0.833
  n_estimators=50 -> R2 = 0.8498
  n_estimators=100 -> R2 = 0.8538
  n_estimators=200 -> R2 = 0.8525
  n_estimators=300 -> R2 = 0.852


In [24]:
# --- Experiment 2: max_features ---
print("=== RF Experiment 2: max_features (n=100) ===")
for mf in [None, 'sqrt', 'log2', 3, 4, 5]:
    m = RandomForestRegressor(n_estimators=100, max_features=mf, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  max_features={mf} -> R2 = {s}")

=== RF Experiment 2: max_features (n=100) ===
  max_features=None -> R2 = 0.8538
  max_features=sqrt -> R2 = 0.871
  max_features=log2 -> R2 = 0.871
  max_features=3 -> R2 = 0.868
  max_features=4 -> R2 = 0.8581
  max_features=5 -> R2 = 0.8538


In [25]:
# --- Experiment 3: max_depth ---
print("=== RF Experiment 3: max_depth (n=100, max_features=sqrt) ===")
for d in [5, 10, 15, 20, None]:
    m = RandomForestRegressor(n_estimators=100, max_features='sqrt', max_depth=d, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  max_depth={d} -> R2 = {s}")

=== RF Experiment 3: max_depth (n=100, max_features=sqrt) ===
  max_depth=5 -> R2 = 0.8797
  max_depth=10 -> R2 = 0.8776
  max_depth=15 -> R2 = 0.8727
  max_depth=20 -> R2 = 0.871
  max_depth=None -> R2 = 0.871


In [26]:
# --- Experiment 4: min_samples_split ---
print("=== RF Experiment 4: min_samples_split (n=100, max_features=sqrt, max_depth=5) ===")
for mss in [2, 5, 10, 15]:
    m = RandomForestRegressor(n_estimators=100, max_features='sqrt', max_depth=5 ,min_samples_split=mss, random_state=0)
    m.fit(X_train, y_train)
    s = round(r2_score(y_test, m.predict(X_test)), 4)
    print(f"  min_samples_split={mss} -> R2 = {s}")

=== RF Experiment 4: min_samples_split (n=100, max_features=sqrt, max_depth=5) ===
  min_samples_split=2 -> R2 = 0.8797
  min_samples_split=5 -> R2 = 0.8794
  min_samples_split=10 -> R2 = 0.8729
  min_samples_split=15 -> R2 = 0.8745


In [27]:
# Best Random Forest: n_estimators=100, max_features='sqrt'
best_rf = RandomForestRegressor(n_estimators=100, max_features='sqrt',max_depth=5 ,min_samples_split=2,random_state=0)
best_rf.fit(X_train, y_train)
score_rf = round(r2_score(y_test, best_rf.predict(X_test)), 4)
print("Best Random Forest R2 Score:", score_rf)

pickle.dump(best_rf, open("best_random_forest.sav", "wb"))
print("Random Forest model saved.")

Best Random Forest R2 Score: 0.8797
Random Forest model saved.


In [28]:
# =============================================================
# FINAL COMPARISON - All Algorithm Best Scores
# =============================================================
print("=" * 50)
print("ALGORITHM COMPARISON SUMMARY")
print("=" * 50)
print(f"  Linear Regression  : R2 = {score_lr}")
print(f"  SVR                : R2 = {score_svr}")
print(f"  Decision Tree      : R2 = {score_dt}")
print(f"  Random Forest (n_estimators=100, max_features='sqrt',max_depth=5 ,min_samples_split=2,random_state=0)      : R2 = {score_rf}  ← BEST")
print("=" * 50)
print("\nFINAL BEST MODEL: RF with n_estimators=100, max_features='sqrt',max_depth=5 ,min_samples_split=2")
print("Reason: Highest R2 score across all algorithms tested")

ALGORITHM COMPARISON SUMMARY
  Linear Regression  : R2 = 0.7895
  SVR                : R2 = 0.878
  Decision Tree      : R2 = 0.8778
  Random Forest (n_estimators=100, max_features='sqrt',max_depth=5 ,min_samples_split=2,random_state=0)      : R2 = 0.8797  ← BEST

FINAL BEST MODEL: RF with n_estimators=100, max_features='sqrt',max_depth=5 ,min_samples_split=2
Reason: Highest R2 score across all algorithms tested


In [29]:
# =============================================================
# FINAL MODEL - Deployment 
# =============================================================

# Load models
loaded_rf    = pickle.load(open("best_random_forest.sav", "rb"))

# Sample input: age=35, bmi=28.5, children=2, sex_male=0 (female), smoker_yes=1
# Columns order: age, bmi, children, sex_male, smoker_yes
sample_input = [[35, 28.5, 2, 0, 1]]

prediction    = loaded_rf.predict(sample_input)

print("=== PREDICTION WITH BEST MODEL (RF) ===")
print("Input: age=35, bmi=28.5, 2 children, female, smoker")
print(f"Predicted Insurance Charge: ${prediction[0]:,.2f}")

=== PREDICTION WITH BEST MODEL (RF) ===
Input: age=35, bmi=28.5, 2 children, female, smoker
Predicted Insurance Charge: $20,789.81
